# DSA 4060 Personalized Movie Recommender

**Name:** Nicholas Kinyanjui  
**Student ID:** 670178  
**Assigned User ID:** 39

Assigned User ID calculation: last two digits are `78`; `78 MOD 40 = 38`; therefore Assigned User ID is `39`.

## Task 1 - Load and Inspect the Data

This section loads the supplied movie catalogue and user ratings, then checks the structure and missing values.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_colwidth', 80)

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print('First five rows of movies.csv')
display(movies.head())

print('First five rows of ratings.csv')
display(ratings.head())

First five rows of movies.csv


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


First five rows of ratings.csv


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [2]:
print(f'movies.csv shape: {movies.shape[0]} rows and {movies.shape[1]} columns')
print(f'ratings.csv shape: {ratings.shape[0]} rows and {ratings.shape[1]} columns')

movies.csv shape: 36 rows and 3 columns
ratings.csv shape: 440 rows and 3 columns


In [3]:
missing_summary = pd.DataFrame({
    'movies_missing_values': movies.isna().sum(),
}).join(pd.DataFrame({'ratings_missing_values': ratings.isna().sum()}), how='outer').fillna(0).astype(int)

display(missing_summary)
print('Finding: Both datasets contain 0 missing values in their columns.')

,movies_missing_values,ratings_missing_values
genres,0,0
movie_id,0,0
rating,0,0
title,0,0
user_id,0,0


Finding: Both datasets contain 0 missing values in their columns.


## Task 2 - Profile the Assigned User

The assigned user is User ID 39.

In [4]:
assigned_user_id = 39

user_ratings = (
    ratings[ratings['user_id'] == assigned_user_id]
    .merge(movies, on='movie_id', how='left')
    [['user_id', 'movie_id', 'title', 'genres', 'rating']]
    .sort_values(['rating', 'title'], ascending=[False, True])
)

display(user_ratings)

,user_id,movie_id,title,genres,rating
7,39,30,The Grand Budapest Hotel,Comedy|Drama,4.5
2,39,20,Hidden Figures,Drama|Biography,4.0
5,39,28,Knives Out,Mystery|Comedy,4.0
6,39,29,Sherlock Holmes,Mystery|Action,4.0
0,39,6,Blade Runner 2049,Sci-Fi|Drama,3.5
9,39,33,Ford v Ferrari,Sports|Drama,3.5
12,39,36,Remember the Titans,Sports|Drama,3.5
1,39,17,The Pursuit of Happyness,Drama|Biography,3.5
8,39,31,The Hangover,Comedy,3.0
10,39,34,Creed,Sports|Drama,2.5


In [5]:
top_three_movies = user_ratings.sort_values(['rating', 'title'], ascending=[False, True]).head(3)
display(top_three_movies[['title', 'genres', 'rating']])

,title,genres,rating
7,The Grand Budapest Hotel,Comedy|Drama,4.5
2,Hidden Figures,Drama|Biography,4.0
5,Knives Out,Mystery|Comedy,4.0


In [6]:
genre_profile = (
    user_ratings.assign(genre=user_ratings['genres'].str.split('|'))
    .explode('genre')
    .groupby('genre')
    .agg(
        rated_movies=('movie_id', 'count'),
        average_rating=('rating', 'mean'),
        high_ratings=('rating', lambda s: int((s >= 4.0).sum()))
    )
    .sort_values(['average_rating', 'high_ratings', 'rated_movies'], ascending=False)
    .round(2)
)

display(genre_profile)

,rated_movies,average_rating,high_ratings
genre,,,
Mystery,2,4.00,2
Action,1,4.00,1
Comedy,3,3.83,2
Biography,2,3.75,1
Sci-Fi,1,3.50,0
Drama,8,3.44,2
Sports,4,3.00,0
Horror,1,2.50,0
Thriller,1,2.50,0


User 39 appears to prefer Mystery, Comedy, Drama, and Biography. Mystery is the clearest preference because both Mystery movies rated by the user received 4.0, while Comedy has two high ratings and a strong average rating of 3.83 across three movies. Drama appears often in the user's history, including two positively rated films, but its lower average rating of 3.44 shows that Drama alone is not always enough. The user's lower scores for Sports, Horror, Thriller, Animation, and Adventure suggest those genres are weaker recommendation signals for this user.

## Task 3 - Build the Recommendation Logic

Movies rated 4.0 or higher are treated as positive evidence. Movie genres are converted into numerical one-hot features, and cosine similarity is used to score unrated movies against the user's liked movies.

In [7]:
liked_movies = user_ratings[user_ratings['rating'] >= 4.0].copy()
display(liked_movies[['title', 'genres', 'rating']])

,title,genres,rating
7,The Grand Budapest Hotel,Comedy|Drama,4.5
2,Hidden Figures,Drama|Biography,4.0
5,Knives Out,Mystery|Comedy,4.0
6,Sherlock Holmes,Mystery|Action,4.0


In [8]:
# Convert pipe-separated genres into numerical one-hot genre features.
all_genres = sorted({genre for genre_list in movies['genres'] for genre in genre_list.split('|')})

genre_features = pd.DataFrame(
    [[1 if genre in genre_list.split('|') else 0 for genre in all_genres] for genre_list in movies['genres']],
    columns=all_genres,
    index=movies['movie_id']
)

display(genre_features.head())

,Action,Adventure,Animation,Biography,Comedy,Crime,Drama,Family,Horror,Musical,Mystery,Romance,Sci-Fi,Sports,Superhero,Thriller
movie_id,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
2,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
5,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [9]:
# Cosine similarity on genre vectors. This avoids recommending movies already rated by the user.
feature_matrix = genre_features.to_numpy(dtype=float)
norms = np.linalg.norm(feature_matrix, axis=1, keepdims=True)
normalized_features = np.divide(feature_matrix, norms, out=np.zeros_like(feature_matrix), where=norms != 0)

liked_movie_ids = liked_movies['movie_id'].tolist()
liked_positions = movies.index[movies['movie_id'].isin(liked_movie_ids)].tolist()

similarity_to_liked = normalized_features @ normalized_features[liked_positions].T
movies_with_scores = movies.copy()
movies_with_scores['similarity_score'] = similarity_to_liked.mean(axis=1)

already_rated_ids = set(user_ratings['movie_id'])
recommendations = (
    movies_with_scores[~movies_with_scores['movie_id'].isin(already_rated_ids)]
    .sort_values(['similarity_score', 'title'], ascending=[False, True])
    .head(5)
    .reset_index(drop=True)
)
recommendations['Rank'] = recommendations.index + 1
recommendations['similarity_score'] = recommendations['similarity_score'].round(3)

display(recommendations[['Rank', 'title', 'genres', 'similarity_score']])
print('Recommended movie titles:')
for title in recommendations['title']:
    print('-', title)

,Rank,title,genres,similarity_score
0,1,The Social Network,Drama|Biography,0.375
1,2,The Shawshank Redemption,Drama,0.354
2,3,Arrival,Sci-Fi|Drama,0.250
3,4,Crazy Rich Asians,Romance|Comedy,0.250
4,5,Interstellar,Sci-Fi|Drama,0.250


Recommended movie titles:
- The Social Network
- The Shawshank Redemption
- Arrival
- Crazy Rich Asians
- Interstellar


## Task 4 - Explain the Recommendations

In [10]:
liked_lookup = liked_movies.set_index('movie_id')[['title', 'genres', 'rating']]

reason_rows = []
for _, rec in recommendations.iterrows():
    rec_position = movies.index[movies['movie_id'] == rec['movie_id']][0]
    rec_similarities = similarity_to_liked[rec_position]
    best_liked_position = int(np.argmax(rec_similarities))
    best_liked_movie_id = liked_movie_ids[best_liked_position]
    best_liked = liked_lookup.loc[best_liked_movie_id]
    shared_genres = sorted(set(rec['genres'].split('|')).intersection(set(best_liked['genres'].split('|'))))
    shared_text = ', '.join(shared_genres) if shared_genres else 'related genre profile'
    reason_rows.append({
        'Rank': int(rec['Rank']),
        'Recommended Movie': rec['title'],
        'Genres': rec['genres'],
        'Reason': f"Shares {shared_text} with '{best_liked['title']}', which User 39 rated {best_liked['rating']:.1f}."
    })

results_table = pd.DataFrame(reason_rows)
display(results_table)

,Rank,Recommended Movie,Genres,Reason
0,1,The Social Network,Drama|Biography,"Shares Drama with 'The Grand Budapest Hotel', which User 39 rated 4.5."
1,2,The Shawshank Redemption,Drama,"Shares Drama with 'The Grand Budapest Hotel', which User 39 rated 4.5."
2,3,Arrival,Sci-Fi|Drama,"Shares Drama with 'The Grand Budapest Hotel', which User 39 rated 4.5."
3,4,Crazy Rich Asians,Romance|Comedy,"Shares related genre profile with 'Hidden Figures', which User 39 rated 4.0."
4,5,Interstellar,Sci-Fi|Drama,"Shares Drama with 'The Grand Budapest Hotel', which User 39 rated 4.5."


## Task 5 - Critical Thinking

**Limitation:** This recommender only uses genres, so it cannot understand other important movie qualities such as actors, directors, release year, plot style, popularity, or how similar users behaved. Because of that, two movies with the same genre labels may be treated as equally relevant even if they feel very different to a viewer.

**Improvement:** A realistic improvement would be to combine this content-based method with collaborative filtering from other users' ratings. That would make the recommendations more personalized because the system could learn from users with similar rating patterns, not only from genre overlap.